# Task 5: Add Logging to the Agentic Workflow

This notebook demonstrates how to add comprehensive logging to monitor tools being used in an agentic workflow. Proper logging is essential for:

- **Debugging**: Understanding what tools are being called and when
- **Monitoring**: Tracking performance and identifying bottlenecks
- **Auditing**: Maintaining a record of all actions taken by the agent
- **Optimization**: Analyzing tool usage patterns to improve workflows

## 1. Setting Up the Logging Infrastructure

First, let's set up a comprehensive logging system that can capture tool usage in agentic workflows.

In [ ]:
import logging
import json
import time
import functools
from datetime import datetime
from typing import Any, Callable, Dict, List, Optional
from dataclasses import dataclass, field, asdict
import sys

# Configure the root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Create a specific logger for tool monitoring
tool_logger = logging.getLogger('agentic_workflow.tools')
tool_logger.setLevel(logging.DEBUG)

print("Logging infrastructure initialized!")

## 2. Creating a Tool Usage Tracker

We'll create a data class to track detailed information about each tool invocation.

In [ ]:
@dataclass
class ToolInvocation:
    """Represents a single tool invocation with all relevant metadata."""
    tool_name: str
    invocation_id: str
    timestamp: str
    input_args: Dict[str, Any]
    output: Any = None
    error: Optional[str] = None
    duration_ms: float = 0.0
    status: str = "pending"
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)
    
    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2, default=str)


class ToolUsageTracker:
    """Tracks all tool invocations during an agentic workflow session."""
    
    def __init__(self, session_id: Optional[str] = None):
        self.session_id = session_id or datetime.now().strftime('%Y%m%d_%H%M%S')
        self.invocations: List[ToolInvocation] = []
        self.logger = logging.getLogger(f'agentic_workflow.tracker.{self.session_id}')
        self._invocation_counter = 0
        
    def _generate_invocation_id(self) -> str:
        self._invocation_counter += 1
        return f"{self.session_id}_{self._invocation_counter:04d}"
    
    def start_invocation(self, tool_name: str, input_args: Dict[str, Any], 
                         metadata: Optional[Dict[str, Any]] = None) -> ToolInvocation:
        """Record the start of a tool invocation."""
        invocation = ToolInvocation(
            tool_name=tool_name,
            invocation_id=self._generate_invocation_id(),
            timestamp=datetime.now().isoformat(),
            input_args=input_args,
            status="running",
            metadata=metadata or {}
        )
        self.invocations.append(invocation)
        
        self.logger.info(
            f"TOOL_START | {invocation.invocation_id} | {tool_name} | "
            f"args={json.dumps(input_args, default=str)}"
        )
        return invocation
    
    def complete_invocation(self, invocation: ToolInvocation, output: Any, 
                            duration_ms: float) -> None:
        """Record the successful completion of a tool invocation."""
        invocation.output = output
        invocation.duration_ms = duration_ms
        invocation.status = "success"
        
        self.logger.info(
            f"TOOL_SUCCESS | {invocation.invocation_id} | {invocation.tool_name} | "
            f"duration={duration_ms:.2f}ms | output_preview={str(output)[:100]}"
        )
    
    def fail_invocation(self, invocation: ToolInvocation, error: str, 
                        duration_ms: float) -> None:
        """Record a failed tool invocation."""
        invocation.error = error
        invocation.duration_ms = duration_ms
        invocation.status = "failed"
        
        self.logger.error(
            f"TOOL_FAILED | {invocation.invocation_id} | {invocation.tool_name} | "
            f"duration={duration_ms:.2f}ms | error={error}"
        )
    
    def get_summary(self) -> Dict[str, Any]:
        """Generate a summary of all tool invocations."""
        tool_stats = {}
        for inv in self.invocations:
            if inv.tool_name not in tool_stats:
                tool_stats[inv.tool_name] = {
                    'total_calls': 0,
                    'success_count': 0,
                    'failure_count': 0,
                    'total_duration_ms': 0.0
                }
            stats = tool_stats[inv.tool_name]
            stats['total_calls'] += 1
            stats['total_duration_ms'] += inv.duration_ms
            if inv.status == 'success':
                stats['success_count'] += 1
            elif inv.status == 'failed':
                stats['failure_count'] += 1
        
        # Calculate averages
        for tool_name, stats in tool_stats.items():
            if stats['total_calls'] > 0:
                stats['avg_duration_ms'] = stats['total_duration_ms'] / stats['total_calls']
        
        return {
            'session_id': self.session_id,
            'total_invocations': len(self.invocations),
            'tool_statistics': tool_stats
        }


# Create a global tracker instance
tracker = ToolUsageTracker()
print(f"Tool Usage Tracker initialized with session ID: {tracker.session_id}")

## 3. Creating a Logging Decorator for Tools

We'll create a decorator that automatically logs tool usage when applied to tool functions.

In [ ]:
def log_tool_usage(tracker: ToolUsageTracker, metadata: Optional[Dict[str, Any]] = None):
    """
    Decorator to automatically log tool usage.
    
    Args:
        tracker: The ToolUsageTracker instance to use for logging
        metadata: Optional metadata to include with each invocation
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            # Capture input arguments
            input_args = {
                'args': args,
                'kwargs': kwargs
            }
            
            # Start tracking
            invocation = tracker.start_invocation(
                tool_name=func.__name__,
                input_args=input_args,
                metadata=metadata
            )
            
            start_time = time.perf_counter()
            
            try:
                result = func(*args, **kwargs)
                duration_ms = (time.perf_counter() - start_time) * 1000
                tracker.complete_invocation(invocation, result, duration_ms)
                return result
            except Exception as e:
                duration_ms = (time.perf_counter() - start_time) * 1000
                tracker.fail_invocation(invocation, str(e), duration_ms)
                raise
        
        return wrapper
    return decorator


print("Tool logging decorator created!")

## 4. Defining Sample Tools with Logging

Let's create some sample tools that an agentic workflow might use, with logging enabled.

In [ ]:
@log_tool_usage(tracker, metadata={'category': 'search'})
def web_search(query: str, max_results: int = 5) -> List[Dict[str, str]]:
    """
    Simulates a web search tool.
    
    Args:
        query: The search query
        max_results: Maximum number of results to return
    
    Returns:
        List of search results
    """
    # Simulate search delay
    time.sleep(0.1)
    
    # Return mock results
    return [
        {'title': f'Result {i} for: {query}', 'url': f'https://example.com/{i}'}
        for i in range(min(max_results, 5))
    ]


@log_tool_usage(tracker, metadata={'category': 'calculation'})
def calculator(expression: str) -> float:
    """
    Evaluates a mathematical expression.
    
    Args:
        expression: A mathematical expression to evaluate
    
    Returns:
        The result of the calculation
    """
    # Simple and safe evaluation for basic math
    allowed_chars = set('0123456789+-*/(). ')
    if not all(c in allowed_chars for c in expression):
        raise ValueError("Invalid characters in expression")
    
    return eval(expression)


@log_tool_usage(tracker, metadata={'category': 'data'})
def fetch_weather(city: str) -> Dict[str, Any]:
    """
    Simulates fetching weather data for a city.
    
    Args:
        city: The city name
    
    Returns:
        Weather data dictionary
    """
    # Simulate API delay
    time.sleep(0.05)
    
    return {
        'city': city,
        'temperature': 72,
        'conditions': 'Sunny',
        'humidity': 45
    }


@log_tool_usage(tracker, metadata={'category': 'file_ops'})
def read_file(filepath: str) -> str:
    """
    Simulates reading a file (with intentional failure for demo).
    
    Args:
        filepath: Path to the file
    
    Returns:
        File contents
    """
    if 'nonexistent' in filepath:
        raise FileNotFoundError(f"File not found: {filepath}")
    
    return f"Contents of {filepath}"


print("Sample tools with logging have been defined!")

## 5. Simulating an Agentic Workflow

Now let's simulate an agentic workflow that uses these tools and observe the logging output.

In [ ]:
def run_agent_workflow(user_query: str) -> str:
    """
    Simulates an agentic workflow that processes a user query.
    
    Args:
        user_query: The user's request
    
    Returns:
        The agent's response
    """
    workflow_logger = logging.getLogger('agentic_workflow.main')
    workflow_logger.info(f"Starting workflow for query: {user_query}")
    
    results = []
    
    # Step 1: Search for information
    workflow_logger.info("Step 1: Performing web search")
    search_results = web_search(user_query, max_results=3)
    results.append(f"Found {len(search_results)} search results")
    
    # Step 2: Get weather data
    workflow_logger.info("Step 2: Fetching weather data")
    weather = fetch_weather("New York")
    results.append(f"Weather in {weather['city']}: {weather['temperature']}F")
    
    # Step 3: Perform a calculation
    workflow_logger.info("Step 3: Performing calculation")
    calc_result = calculator("(10 + 20) * 3")
    results.append(f"Calculation result: {calc_result}")
    
    # Step 4: Try to read a file (will succeed)
    workflow_logger.info("Step 4: Reading file")
    file_content = read_file("config.txt")
    results.append(f"File read: {file_content[:50]}")
    
    # Step 5: Try to read a nonexistent file (will fail - demonstrating error logging)
    workflow_logger.info("Step 5: Attempting to read nonexistent file")
    try:
        read_file("nonexistent.txt")
    except FileNotFoundError:
        results.append("File not found (expected error)")
    
    workflow_logger.info("Workflow completed")
    return "\n".join(results)


# Run the workflow
print("=" * 60)
print("RUNNING AGENTIC WORKFLOW")
print("=" * 60)
print()

response = run_agent_workflow("What's the latest news about AI?")

print()
print("=" * 60)
print("WORKFLOW RESPONSE:")
print("=" * 60)
print(response)

## 6. Analyzing Tool Usage

Let's analyze the logged tool usage data.

In [ ]:
# Get and display the summary
summary = tracker.get_summary()

print("=" * 60)
print("TOOL USAGE SUMMARY")
print("=" * 60)
print(f"\nSession ID: {summary['session_id']}")
print(f"Total Tool Invocations: {summary['total_invocations']}")
print("\nPer-Tool Statistics:")
print("-" * 40)

for tool_name, stats in summary['tool_statistics'].items():
    print(f"\n{tool_name}:")
    print(f"  Total calls: {stats['total_calls']}")
    print(f"  Successes: {stats['success_count']}")
    print(f"  Failures: {stats['failure_count']}")
    print(f"  Avg duration: {stats.get('avg_duration_ms', 0):.2f}ms")

In [ ]:
# Display detailed invocation log
print("=" * 60)
print("DETAILED INVOCATION LOG")
print("=" * 60)

for inv in tracker.invocations:
    status_emoji = "[OK]" if inv.status == 'success' else "[FAIL]"
    print(f"\n{status_emoji} {inv.invocation_id}")
    print(f"   Tool: {inv.tool_name}")
    print(f"   Time: {inv.timestamp}")
    print(f"   Duration: {inv.duration_ms:.2f}ms")
    print(f"   Status: {inv.status}")
    if inv.error:
        print(f"   Error: {inv.error}")

## 7. Advanced: File-Based Logging Handler

For production use, you'll want to persist logs to files. Here's how to set up file-based logging.

In [ ]:
import os

class JSONFileHandler(logging.Handler):
    """
    Custom logging handler that writes structured JSON logs to a file.
    """
    
    def __init__(self, filename: str):
        super().__init__()
        self.filename = filename
        # Ensure directory exists
        os.makedirs(os.path.dirname(filename) if os.path.dirname(filename) else '.', exist_ok=True)
        
    def emit(self, record: logging.LogRecord) -> None:
        try:
            log_entry = {
                'timestamp': datetime.fromtimestamp(record.created).isoformat(),
                'level': record.levelname,
                'logger': record.name,
                'message': record.getMessage(),
                'module': record.module,
                'function': record.funcName,
                'line': record.lineno
            }
            
            with open(self.filename, 'a') as f:
                f.write(json.dumps(log_entry) + '\n')
                
        except Exception:
            self.handleError(record)


def setup_file_logging(log_file: str = 'logs/workflow.jsonl') -> logging.Logger:
    """
    Set up file-based logging for the agentic workflow.
    
    Args:
        log_file: Path to the log file
    
    Returns:
        Configured logger
    """
    logger = logging.getLogger('agentic_workflow')
    
    # Add JSON file handler
    json_handler = JSONFileHandler(log_file)
    json_handler.setLevel(logging.DEBUG)
    logger.addHandler(json_handler)
    
    # Also add a standard file handler for human-readable logs
    file_handler = logging.FileHandler(log_file.replace('.jsonl', '.log'))
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(name)s | %(levelname)s | %(message)s'
    ))
    logger.addHandler(file_handler)
    
    return logger


# Set up file logging
file_logger = setup_file_logging('logs/agentic_workflow.jsonl')
print("File-based logging configured!")
print("Logs will be written to:")
print("  - logs/agentic_workflow.jsonl (structured JSON)")
print("  - logs/agentic_workflow.log (human-readable)")

## 8. Advanced: Callback-Based Tool Monitoring

For more flexible integration with different agent frameworks, here's a callback-based approach.

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol


class ToolCallback(Protocol):
    """Protocol for tool monitoring callbacks."""
    
    def on_tool_start(self, tool_name: str, inputs: Dict[str, Any]) -> None:
        """Called when a tool starts execution."""
        ...
    
    def on_tool_end(self, tool_name: str, outputs: Any) -> None:
        """Called when a tool completes successfully."""
        ...
    
    def on_tool_error(self, tool_name: str, error: Exception) -> None:
        """Called when a tool fails."""
        ...


class LoggingCallback:
    """Callback that logs tool usage to Python logging."""
    
    def __init__(self, logger: Optional[logging.Logger] = None):
        self.logger = logger or logging.getLogger('agentic_workflow.callbacks')
        self._start_times: Dict[str, float] = {}
    
    def on_tool_start(self, tool_name: str, inputs: Dict[str, Any]) -> None:
        self._start_times[tool_name] = time.perf_counter()
        self.logger.info(f"[CALLBACK] Tool '{tool_name}' started with inputs: {inputs}")
    
    def on_tool_end(self, tool_name: str, outputs: Any) -> None:
        duration = (time.perf_counter() - self._start_times.get(tool_name, 0)) * 1000
        self.logger.info(f"[CALLBACK] Tool '{tool_name}' completed in {duration:.2f}ms")
    
    def on_tool_error(self, tool_name: str, error: Exception) -> None:
        duration = (time.perf_counter() - self._start_times.get(tool_name, 0)) * 1000
        self.logger.error(f"[CALLBACK] Tool '{tool_name}' failed after {duration:.2f}ms: {error}")


class MetricsCallback:
    """Callback that collects metrics about tool usage."""
    
    def __init__(self):
        self.metrics: Dict[str, Dict[str, Any]] = {}
        self._start_times: Dict[str, float] = {}
    
    def on_tool_start(self, tool_name: str, inputs: Dict[str, Any]) -> None:
        self._start_times[tool_name] = time.perf_counter()
        if tool_name not in self.metrics:
            self.metrics[tool_name] = {
                'call_count': 0,
                'success_count': 0,
                'error_count': 0,
                'total_duration_ms': 0.0,
                'durations': []
            }
        self.metrics[tool_name]['call_count'] += 1
    
    def on_tool_end(self, tool_name: str, outputs: Any) -> None:
        duration = (time.perf_counter() - self._start_times.get(tool_name, 0)) * 1000
        self.metrics[tool_name]['success_count'] += 1
        self.metrics[tool_name]['total_duration_ms'] += duration
        self.metrics[tool_name]['durations'].append(duration)
    
    def on_tool_error(self, tool_name: str, error: Exception) -> None:
        duration = (time.perf_counter() - self._start_times.get(tool_name, 0)) * 1000
        self.metrics[tool_name]['error_count'] += 1
        self.metrics[tool_name]['total_duration_ms'] += duration
        self.metrics[tool_name]['durations'].append(duration)
    
    def get_report(self) -> Dict[str, Any]:
        """Generate a metrics report."""
        report = {}
        for tool_name, m in self.metrics.items():
            report[tool_name] = {
                'call_count': m['call_count'],
                'success_rate': m['success_count'] / m['call_count'] if m['call_count'] > 0 else 0,
                'avg_duration_ms': m['total_duration_ms'] / m['call_count'] if m['call_count'] > 0 else 0,
                'min_duration_ms': min(m['durations']) if m['durations'] else 0,
                'max_duration_ms': max(m['durations']) if m['durations'] else 0
            }
        return report


# Create callback instances
logging_callback = LoggingCallback()
metrics_callback = MetricsCallback()

print("Callback-based monitoring system created!")

In [ ]:
def make_tool_with_callbacks(func: Callable, callbacks: List[ToolCallback]) -> Callable:
    """
    Wrap a tool function with multiple callbacks.
    
    Args:
        func: The tool function to wrap
        callbacks: List of callback objects
    
    Returns:
        Wrapped function
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        inputs = {'args': args, 'kwargs': kwargs}
        
        # Notify all callbacks of start
        for cb in callbacks:
            cb.on_tool_start(func.__name__, inputs)
        
        try:
            result = func(*args, **kwargs)
            # Notify all callbacks of success
            for cb in callbacks:
                cb.on_tool_end(func.__name__, result)
            return result
        except Exception as e:
            # Notify all callbacks of error
            for cb in callbacks:
                cb.on_tool_error(func.__name__, e)
            raise
    
    return wrapper


# Example: Create a tool with multiple callbacks
def raw_database_query(query: str) -> List[Dict]:
    """Simulates a database query."""
    time.sleep(0.05)  # Simulate query time
    return [{'id': 1, 'name': 'Test', 'value': query}]


# Wrap with callbacks
database_query = make_tool_with_callbacks(
    raw_database_query, 
    [logging_callback, metrics_callback]
)

# Use the tool
print("\nExecuting database queries with callback monitoring:")
print("-" * 50)

for i in range(3):
    result = database_query(f"SELECT * FROM table_{i}")
    print(f"Query {i+1} returned {len(result)} results")

print("\n" + "-" * 50)
print("Metrics Report:")
print(json.dumps(metrics_callback.get_report(), indent=2))

## 9. Integration Example: LangChain-Style Callback Handler

Here's an example of how this logging system could integrate with popular frameworks like LangChain.

In [ ]:
class AgentCallbackHandler:
    """
    A callback handler compatible with common agent frameworks.
    Provides hooks for various agent lifecycle events.
    """
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.logger = logging.getLogger('agentic_workflow.agent')
        self.run_history: List[Dict[str, Any]] = []
        self._current_run: Optional[Dict[str, Any]] = None
    
    def on_agent_start(self, agent_name: str, input_text: str) -> None:
        """Called when an agent starts processing."""
        self._current_run = {
            'agent': agent_name,
            'input': input_text,
            'start_time': datetime.now().isoformat(),
            'tool_calls': [],
            'llm_calls': []
        }
        if self.verbose:
            self.logger.info(f"Agent '{agent_name}' started with input: {input_text[:100]}...")
    
    def on_tool_start(self, tool_name: str, tool_input: str) -> None:
        """Called when a tool is invoked."""
        tool_call = {
            'tool': tool_name,
            'input': tool_input,
            'start_time': time.perf_counter()
        }
        if self._current_run:
            self._current_run['tool_calls'].append(tool_call)
        if self.verbose:
            self.logger.info(f"Tool '{tool_name}' invoked with: {tool_input[:100]}...")
    
    def on_tool_end(self, tool_name: str, tool_output: str) -> None:
        """Called when a tool completes."""
        if self._current_run and self._current_run['tool_calls']:
            tool_call = self._current_run['tool_calls'][-1]
            tool_call['output'] = tool_output
            tool_call['duration_ms'] = (time.perf_counter() - tool_call['start_time']) * 1000
        if self.verbose:
            self.logger.info(f"Tool '{tool_name}' completed: {tool_output[:100]}...")
    
    def on_llm_start(self, prompt: str) -> None:
        """Called when an LLM is invoked."""
        llm_call = {
            'prompt_length': len(prompt),
            'start_time': time.perf_counter()
        }
        if self._current_run:
            self._current_run['llm_calls'].append(llm_call)
        if self.verbose:
            self.logger.debug(f"LLM invoked with prompt of {len(prompt)} characters")
    
    def on_llm_end(self, response: str) -> None:
        """Called when an LLM completes."""
        if self._current_run and self._current_run['llm_calls']:
            llm_call = self._current_run['llm_calls'][-1]
            llm_call['response_length'] = len(response)
            llm_call['duration_ms'] = (time.perf_counter() - llm_call['start_time']) * 1000
        if self.verbose:
            self.logger.debug(f"LLM responded with {len(response)} characters")
    
    def on_agent_end(self, output: str) -> None:
        """Called when an agent completes."""
        if self._current_run:
            self._current_run['end_time'] = datetime.now().isoformat()
            self._current_run['output'] = output
            self.run_history.append(self._current_run)
            self._current_run = None
        if self.verbose:
            self.logger.info(f"Agent completed with output: {output[:100]}...")
    
    def get_run_summary(self) -> Dict[str, Any]:
        """Get a summary of all agent runs."""
        total_tool_calls = sum(len(run['tool_calls']) for run in self.run_history)
        total_llm_calls = sum(len(run['llm_calls']) for run in self.run_history)
        
        return {
            'total_runs': len(self.run_history),
            'total_tool_calls': total_tool_calls,
            'total_llm_calls': total_llm_calls,
            'runs': self.run_history
        }


# Demo the callback handler
handler = AgentCallbackHandler(verbose=True)

print("\nSimulating agent execution with callback handler:")
print("=" * 60)

# Simulate an agent run
handler.on_agent_start("ResearchAgent", "Find information about Python logging best practices")
handler.on_llm_start("What tools should I use to find information about Python logging?")
handler.on_llm_end("I should use the web_search tool to find relevant articles.")
handler.on_tool_start("web_search", "Python logging best practices 2024")
time.sleep(0.1)  # Simulate tool execution
handler.on_tool_end("web_search", "Found 5 relevant articles about Python logging")
handler.on_llm_start("Based on the search results, summarize the best practices.")
handler.on_llm_end("The key best practices are: use proper log levels, structured logging, etc.")
handler.on_agent_end("Here are the Python logging best practices: ...")

print("\n" + "=" * 60)
print("Run Summary:")
print(json.dumps(handler.get_run_summary(), indent=2, default=str))

## 10. Summary and Best Practices

This notebook demonstrated several approaches to adding logging to agentic workflows:

### Key Components Implemented:

1. **ToolInvocation dataclass**: Structured representation of tool calls
2. **ToolUsageTracker**: Central tracker for all tool invocations
3. **@log_tool_usage decorator**: Easy-to-use decorator for automatic logging
4. **JSONFileHandler**: File-based logging with structured JSON output
5. **Callback-based monitoring**: Flexible callback system for different frameworks
6. **AgentCallbackHandler**: Framework-compatible callback handler

### Best Practices:

- **Use structured logging**: JSON logs are easier to parse and analyze
- **Track timing information**: Duration metrics help identify bottlenecks
- **Log at appropriate levels**: Use DEBUG for detailed info, INFO for normal operations, ERROR for failures
- **Include context**: Add metadata like session IDs and invocation IDs for traceability
- **Handle errors gracefully**: Always log errors with full context
- **Generate summaries**: Periodic summaries help monitor overall system health

In [ ]:
# Final summary of everything we've tracked
print("=" * 60)
print("FINAL SESSION SUMMARY")
print("=" * 60)

final_summary = tracker.get_summary()
print(f"\nSession ID: {final_summary['session_id']}")
print(f"Total tool invocations tracked: {final_summary['total_invocations']}")

print("\nTools used in this session:")
for tool_name, stats in final_summary['tool_statistics'].items():
    success_rate = (stats['success_count'] / stats['total_calls'] * 100) if stats['total_calls'] > 0 else 0
    print(f"  - {tool_name}: {stats['total_calls']} calls, {success_rate:.0f}% success rate")

print("\nLogging infrastructure is now ready for production use!")